# AI 08 - Ev tapşırığı
# Avtomobilin qiymətinin proqnozlaşdırılması - Decision Tree

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
FILE_PATH = 'Used_Cars_Homework.csv'
data = pd.read_csv(FILE_PATH)

# 1. İlkin təhlil (EDA)

In [ ]:
data.head(10)

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data.isna().sum()

In [ ]:
data.describe()

In [ ]:
print('Min qiymət:', data['price_azn'].min())
print('Max qiymət:', data['price_azn'].max())
print('Orta qiymət:', round(data['price_azn'].mean(), 2))

# 2. Vizuallaşdırma

In [ ]:
plt.figure()
plt.hist(data['price_azn'], bins=30)
plt.title('Histogram of price')
plt.xlabel('Price (AZN)')
plt.ylabel('Frequency')
plt.show()

Histoqram qiymətlərin hansı aralıqda toplandığını göstərir, çox ucuz və çox baha maşınlar azdır.

In [ ]:
plt.scatter(data['mileage_km'], data['price_azn'])
plt.title('Price vs mileage')
plt.xlabel('Mileage (km)')
plt.ylabel('Price (AZN)')
plt.show()

Yürüş artdıqca qiymət ümumilikdə azalır.

In [ ]:
plt.scatter(data['age_years'], data['price_azn'])
plt.title('Price vs age')
plt.xlabel('Age (years)')
plt.ylabel('Price (AZN)')
plt.show()

Maşın köhnəldikcə qiyməti düşür.

In [ ]:
brand_mean = data.groupby('brand')['price_azn'].mean().sort_values(ascending=False)

plt.bar(brand_mean.index, brand_mean.values, color='skyblue')
plt.title('Average price by brand')
plt.xlabel('Brand')
plt.ylabel('Average price (AZN)')
plt.show()

Orta qiymətə görə BMW digər markalardan bahadır.

# 3. Verilənlərin hazırlanması

In [ ]:
target_col = 'price_azn'

data_fe = data.copy()
# yas 0 olsa bolme xetasi olmasin deye 1 ile evez edirem
data_fe['mileage_per_year'] = data_fe['mileage_km'] / data_fe['age_years'].replace(0, 1)

numeric_cols = ['age_years', 'mileage_km', 'engine_l', 'power_hp', 'mileage_per_year']
categorical_cols = ['brand', 'fuel_type', 'transmission', 'accident_history']

In [ ]:
X = data_fe.drop(columns=[target_col])
y = data_fe[target_col]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
len(X_train), len(X_test)

## Factorize və One-Hot Encoding (brand)

In [ ]:
demo = X_train.copy()

codes, uniques = pd.factorize(demo['brand'])
demo['brand_factorized'] = codes
print(list(uniques))
demo[['brand', 'brand_factorized']].head(10)

In [ ]:
pd.get_dummies(demo['brand'], prefix='brand').head(10)

## MinMaxScaler və StandardScaler (mileage_km)

In [ ]:
mileage = demo[['mileage_km']].dropna()

mms = MinMaxScaler()
std = StandardScaler()

scaled = pd.DataFrame({
    'mileage_km': mileage['mileage_km'].values,
    'minmax': mms.fit_transform(mileage).ravel(),
    'standard': std.fit_transform(mileage).ravel()
})
scaled.describe().round(3)

**Fərqlər:**
- `factorize` hər markaya bir rəqəm verir, bir sütun qalır, amma saxta sıra yaranır. OHE hər marka üçün ayrı 0/1 sütun yaradır.
- `MinMaxScaler` dəyərləri [0, 1] aralığına salır. `StandardScaler` ortanı 0, standart kənarlaşmanı 1 edir.

Ağac əsaslı modellər üçün StandardScaler adətən lazım deyil, amma tapşırıqda eyni emal ardıcıllığı saxlanılır.

## Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

def make_model(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

# 4. Modelin öyrədilməsi və qiymətləndirilməsi

In [ ]:
final_model = make_model(DecisionTreeRegressor(max_depth=6, random_state=42))

# fit yalniz train datada
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'''
MAE: {mae:.2f}
MSE: {mse:.2f}
RMSE: {rmse:.2f}
R2: {r2:.2f}
''')

**Şərh:**
- **MAE** - proqnoz orta hesabla real qiymətdən bu qədər **AZN** fərqlənir.
- **MSE** - vahidi **AZN²**-dir, böyük xətaları daha çox cəzalandırır.
- **RMSE** - vahidi **AZN**-dir. MAE-dən xeyli böyükdürsə, bəzi maşınlarda böyük xətalar var.
- **R²** - vahidsizdir, model qiymətin dəyişkənliyinin nə qədərini izah edir. 1-ə yaxın olması yaxşıdır.

In [ ]:
result = pd.DataFrame({
    'real_price': y_test.values[:10],
    'predicted': y_pred[:10].round(2),
})
result['abs_error'] = (result['real_price'] - result['predicted']).abs().round(2)
result

# 5. Öyrənmə əyrisi (Learning Curve)
Yalnız train datasından istifadə edirəm.

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    final_model,
    X_train,
    y_train,
    cv=5,
    scoring='r2',
    train_sizes=np.linspace(.1, 1.0, 5),
    shuffle=True,
    random_state=42)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_mean, label='Training R2', marker='o')
plt.plot(train_sizes, test_mean, 'o-', label='CV R2')
plt.title('Learning Curve (Decision Tree)')
plt.ylabel('R2 score')
plt.xlabel('Training set size')
plt.legend(loc='best')
plt.show()

In [ ]:
print('Train R2:', round(train_mean[-1], 3))
print('CV R2:', round(test_mean[-1], 3))
print('Fərq:', round(train_mean[-1] - test_mean[-1], 3))

Nümunə sayı artdıqca CV R² yaxşılaşır. Train ilə CV arasındakı fərq nə qədər böyükdürsə, overfitting ehtimalı o qədər çoxdur.

# 6. Yeni avtomobillər üçün proqnoz

In [ ]:
new_cars = pd.DataFrame({
    'age_years': [5, 10, 3],
    'mileage_km': [75000, 180000, 40000],
    'engine_l': [1.6, 2.0, 2.5],
    'power_hp': [110, 140, 170],
    'brand': ['Toyota', 'Hyundai', 'BMW'],
    'fuel_type': ['petrol', 'diesel', 'petrol'],
    'transmission': ['automatic', 'manual', 'automatic'],
    'accident_history': ['no', 'yes', 'no']
})
new_cars['mileage_per_year'] = new_cars['mileage_km'] / new_cars['age_years'].replace(0, 1)

new_cars['predicted_price'] = final_model.predict(new_cars).round(2)
new_cars

# 7. max_depth-in təsiri
Test datasına toxunmuram, müqayisə train və CV ilə edilir.

In [ ]:
rows = []
for d in [2, 3, 4, 5, 6, 8, 10, 15, None]:
    m = make_model(DecisionTreeRegressor(max_depth=d, random_state=42))
    m.fit(X_train, y_train)
    rows.append({
        'max_depth': str(d),
        'train_r2': r2_score(y_train, m.predict(X_train)),
        'cv_r2': cross_val_score(m, X_train, y_train, cv=5, scoring='r2').mean()
    })
params = pd.DataFrame(rows).round(3)
params

In [ ]:
plt.plot(params['max_depth'], params['train_r2'], marker='o', label='Train R2')
plt.plot(params['max_depth'], params['cv_r2'], marker='o', label='CV R2')
plt.title('Effect of max_depth')
plt.xlabel('max_depth')
plt.ylabel('R2 score')
plt.legend()
plt.show()

Dərinlik artdıqca train R² 1-ə yaxınlaşır, çünki ağac datanı əzbərləyir. Train ilə CV arasındakı fərq böyüdükcə overfitting də artır.

## Ən vacib əlamətlər

In [ ]:
feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()
importances = pd.Series(final_model.named_steps['model'].feature_importances_, index=feature_names)
top = importances.sort_values(ascending=False).head(10)
top

In [ ]:
plt.barh(top.index, top.values, color='skyblue')
plt.gca().invert_yaxis()
plt.title('Top 10 feature importances')
plt.xlabel('Importance')
plt.show()

# 8. Nəticə

In [ ]:
print(f'''
Decision Tree
R2: {r2:.2f}
MAE: {mae:.2f} AZN
MSE: {mse:.2f} AZN^2
RMSE: {rmse:.2f} AZN
''')

Qərar ağacı sadədir və asan başa düşülür, amma tək ağac olduğu üçün overfitting-ə meyllidir. Ona görə `max_depth` ilə dərinliyi məhdudlaşdırmaq lazımdır.

Learning curve-də train və CV fərqinə baxaraq overfitting-i qiymətləndirirəm. Bütün notebooklarda bölgü eyni olduğu üçün bu nəticələri digər modellərlə müqayisə etmək olar.